In [19]:
import numpy as np
import pandas as pd

In [20]:
df = pd.read_csv('./Products.csv')
df.head(3)

,ProductKey,Product Name,Brand,Color,Unit Cost USD,Unit Price USD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,$6.62,$12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,$6.62,$12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,$7.40,$14.52,101,MP4&MP3,1,Audio


In [21]:
df = df.dropna()

In [22]:
main_feature4m_df = df[['ProductKey','Product Name', 'Brand', 'Color','Subcategory', 'Category']]

In [23]:
main_feature4m_df.dtypes

ProductKey       int64
Product Name    object
Brand           object
Color           object
Subcategory     object
Category        object
dtype: object

In [24]:
import re

main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('MP4&MP3', 'MP4 and MP3')
main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace(' & ', ' and ')
main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('Printers, Scanners and Fax', 'Printers Scanners and Fax')

C:\Users\HP\AppData\Local\Temp\ipykernel_452\1586299548.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('MP4&MP3', 'MP4 and MP3')
C:\Users\HP\AppData\Local\Temp\ipykernel_452\1586299548.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace(' & ', ' and ')
C:\Users\HP\AppData\Local\Temp\ipykernel_452\1586299548.py:5: SettingWithCopyWarning: 
A value

In [25]:
main_feature4m_df.head(3)

,ProductKey,Product Name,Brand,Color,Subcategory,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,MP4 and MP3,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,MP4 and MP3,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,MP4 and MP3,Audio


In [26]:
main_feature4m_df['product_details'] = (
    main_feature4m_df['Product Name'] + ' ' + 
    main_feature4m_df['Brand'] + ' ' + 
    main_feature4m_df['Color'] + ' ' + 
    main_feature4m_df['Subcategory'] + ' ' + 
    main_feature4m_df['Category'] 
)

C:\Users\HP\AppData\Local\Temp\ipykernel_452\3394763485.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['product_details'] = (


In [27]:
main_feature4m_df.head(3)

,ProductKey,Product Name,Brand,Color,Subcategory,Category,product_details
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,MP4 and MP3,Audio,Contoso 512MB MP3 Player E51 Silver Contoso Si...
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,MP4 and MP3,Audio,Contoso 512MB MP3 Player E51 Blue Contoso Blue...
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,MP4 and MP3,Audio,Contoso 1G MP3 Player E100 White Contoso White...


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(stop_words='english',max_df=0.95,min_df=2,ngram_range=(1,1))
tfidf_matrix = tfidf_vec.fit_transform(main_feature4m_df['product_details'])

In [29]:
# tfidf_matrix is a sparse matrix means it has lot of zeros in it.
tfidf_matrix.shape

(2517, 980)

In [30]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

def content_based_recommendations(product_id, data, cosine_sim, n=10):
    idx = data[data['ProductKey'] == product_id].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    product_indices = [i[0] for i in sim_scores]
    return data.iloc[product_indices], sim_scores

input_product_id = int(input("Nhập mã sản phẩm: "))  
recommendations_result, sim_score = content_based_recommendations(input_product_id, main_feature4m_df, cosine_sim)

In [32]:
# Hiển thị kết quả
print("Sản phẩm gợi ý:")
print(recommendations_result[['ProductKey', 'Product Name']]) 

print("\nĐiểm độ tương đồng dự đoán:")
print([round(tup[1], 3) for tup in sim_score])  

Sản phẩm gợi ý:
    ProductKey                                Product Name
48          49          WWI 2GB Pulse Smart pen M100 White
51          52         WWI 2GB Pulse Smart pen M100 Silver
50          51           WWI 2GB Pulse Smart pen M100 Blue
46          47            WWI 1GBPulse Smart pen E50 Black
60          61   WWI 2GB Spy Video Recorder Pen M300 Black
45          46           WWI 1GB Pulse Smart pen E50 White
47          48          WWI 1GB Pulse Smart pen E50 Silver
52          53      WWI 4GB Video Recording Pen X200 Black
61          62   WWI 2GB Spy Video Recorder Pen M300 White
63          64  WWI 2GB Spy Video Recorder Pen M300 Silver

Điểm độ tương đồng dự đoán:
[0.931, 0.927, 0.905, 0.763, 0.728, 0.724, 0.721, 0.691, 0.663, 0.66]
